# Truth Through Debate — Multi-Agent LLM Fact Checking
### Deep Learning Course Project

**Before running:** Add your free Groq API key to Colab Secrets:
1. Click the key icon in the left sidebar
2. Click **Add new secret**
3. Name: `truth_through_debate`, Value: your key from console.groq.com
4. Toggle **Notebook access** ON

**5 models used (all free on Groq):**
| Role | Model | Company |
|---|---|---|
| Baseline | llama-3.3-70b-versatile | Meta |
| Debater A (pro-true) | llama-4-scout-17b-16e-instruct | Meta |
| Debater B (pro-false) | llama-3.1-8b-instant | Meta |
| Judge | qwen/qwen3-32b | Alibaba |
| Scorer | moonshotai/kimi-k2-instruct | Moonshot AI |

In [ ]:
# Cell 1 — Install dependencies
!pip install groq datasets pandas numpy scikit-learn tqdm rich python-dotenv pyarrow -q --upgrade

In [ ]:
# Cell 2 — Upload and extract project zip
from google.colab import files
uploaded = files.upload()  # Upload truth_through_debate_v2.zip

import zipfile, sys
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

sys.path.insert(0, '/content/truth_through_debate')
print('Project loaded!')

In [ ]:
# Cell 3 — Load API key and verify all 5 models
import os
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('truth_through_debate')
print('API key loaded!')

from utils.llm_client import BASELINE_MODEL, DEBATER_A_MODEL, DEBATER_B_MODEL, JUDGE_MODEL, SCORER_MODEL
print('\n5 Model Lineup:')
print(f'  Baseline  (single LLM control) : {BASELINE_MODEL}')
print(f'  Debater A (argues TRUE)        : {DEBATER_A_MODEL}')
print(f'  Debater B (argues FALSE)       : {DEBATER_B_MODEL}')
print(f'  Judge     (final verdict)      : {JUDGE_MODEL}')
print(f'  Scorer    (evaluates quality)  : {SCORER_MODEL}')

In [ ]:
# Cell 4 — Quick single-claim test (confirm everything works)
from evaluation.evaluator import run_baseline, run_debate_system

claim = 'The Great Wall of China is visible from space with the naked eye.'
gt    = 'REFUTES'

print('Testing baseline...')
b = run_baseline(claim, gt)
print(f'  Verdict: {b["verdict"]} | Confidence: {b["confidence"]:.2f} | Correct: {b["correct"]}')

print('\nTesting debate system...')
d = run_debate_system(claim, gt, num_rounds=2)
print(f'  Verdict: {d["verdict"]} | Confidence: {d["confidence"]:.2f} | Correct: {d["correct"]}')
print(f'  Explanation: {d["explanation"]}')

In [ ]:
# Cell 5 — Verbose single-claim debate transcript
from debate.engine import run_debate

claim  = 'The Great Wall of China is visible from space with the naked eye.'
result = run_debate(claim, ground_truth='REFUTES', num_rounds=2, verbose=False)

print('=' * 65)
print('  STEP 1 — EVIDENCE  (Llama 4 Scout)')
print('=' * 65)
for i, ev in enumerate(result.evidence, 1):
    print(f'  [{i}] {ev}')

for rd in result.debate_rounds:
    r = rd['round']
    print(f'\n{"-" * 65}')
    print(f'  ROUND {r} — DEBATER A  (Llama 4 Scout)  →  argues TRUE')
    print(f'{"-" * 65}')
    for line in rd['argument_a'].strip().splitlines():
        print(f'  {line}')
    print(f'\n{"-" * 65}')
    print(f'  ROUND {r} — DEBATER B  (Llama 3.1 8B)   →  argues FALSE')
    print(f'{"-" * 65}')
    for line in rd['argument_b'].strip().splitlines():
        print(f'  {line}')

print(f'\n{"=" * 65}')
print('  JUDGE  (Qwen3 32B)  —  Final Verdict')
print('=' * 65)
print(f'  Verdict    : {result.verdict}')
print(f'  Confidence : {result.confidence:.2f}')
print(f'  Correct    : {result.correct}')
print(f'  Explanation: {result.explanation}')

In [ ]:
# Cell 6 — Full experiment on sample claims (fast, no download needed)
from evaluation.evaluator import evaluate_system
from data.sample_claims import load_sample

claims = load_sample(n=20)
print(f'Running experiment on {len(claims)} claims...')
print('Estimated time: ~20 minutes (8s pause between claims for rate limits)\n')

results = evaluate_system(
    claims=claims,
    system='both',
    num_rounds=2,
    score_reasoning=True,
    score_hallucination=True,
    sleep_between=8,
)

bm = results['baseline_metrics']
dm = results['debate_metrics']

print(f'\n{"Metric":<30} {"Baseline":>12} {"Debate":>12} {"Delta":>12}')
print('-' * 68)
print(f'{"Accuracy":<30} {bm["accuracy"]:>12.3f} {dm["accuracy"]:>12.3f} {dm["accuracy"]-bm["accuracy"]:>+12.3f}')
print(f'{"Reasoning Quality":<30} {bm["avg_reasoning_quality"]:>12.2f} {dm["avg_reasoning_quality"]:>12.2f} {dm["avg_reasoning_quality"]-bm["avg_reasoning_quality"]:>+12.2f}')
print(f'{"ECE (lower=better)":<30} {bm["ece"]:>12.4f} {dm["ece"]:>12.4f} {dm["ece"]-bm["ece"]:>+12.4f}')
print(f'{"Hallucination Rate":<30} {bm["hallucination_rate"]:>12.3f} {dm["hallucination_rate"]:>12.3f} {dm["hallucination_rate"]-bm["hallucination_rate"]:>+12.3f}')

In [ ]:
# Cell 7 — Full experiment on FEVER dataset
# Run Cell 1 with runtime restart if this gives an ArrowKeyError
from data.fever_loader import load_fever
from evaluation.evaluator import evaluate_system

fever_claims = load_fever(split='test', n=50, exclude_nei=True)
print(f'Running experiment on {len(fever_claims)} FEVER claims...')

fever_results = evaluate_system(
    claims=fever_claims,
    system='both',
    num_rounds=2,
    score_reasoning=True,
    score_hallucination=True,
    sleep_between=8,
)

bm = fever_results['baseline_metrics']
dm = fever_results['debate_metrics']

print(f'\n{"Metric":<30} {"Baseline":>12} {"Debate":>12} {"Delta":>12}')
print('-' * 68)
print(f'{"Accuracy":<30} {bm["accuracy"]:>12.3f} {dm["accuracy"]:>12.3f} {dm["accuracy"]-bm["accuracy"]:>+12.3f}')
print(f'{"Reasoning Quality":<30} {bm["avg_reasoning_quality"]:>12.2f} {dm["avg_reasoning_quality"]:>12.2f} {dm["avg_reasoning_quality"]-bm["avg_reasoning_quality"]:>+12.2f}')
print(f'{"ECE (lower=better)":<30} {bm["ece"]:>12.4f} {dm["ece"]:>12.4f} {dm["ece"]-bm["ece"]:>+12.4f}')
print(f'{"Hallucination Rate":<30} {bm["hallucination_rate"]:>12.3f} {dm["hallucination_rate"]:>12.3f} {dm["hallucination_rate"]-bm["hallucination_rate"]:>+12.3f}')

In [ ]:
# Cell 8 — Export results to CSV
import pandas as pd

# Use whichever results you have (fever_results or results)
res = fever_results  # change to 'results' if using sample claims

baseline_df = pd.DataFrame(res['baseline_results'])
debate_df   = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'debate_rounds'}
    for r in res['debate_results']
])

baseline_df.to_csv('baseline_results.csv', index=False)
debate_df.to_csv('debate_results.csv', index=False)
print('Saved: baseline_results.csv and debate_results.csv')

In [ ]:
# Cell 9 — Download CSVs
from google.colab import files
files.download('baseline_results.csv')
files.download('debate_results.csv')